In [1]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import os
import leidenalg

In [ ]:
import importlib.metadata as md

packages = ["scanpy", "scvi-tools", "numpy", 
            "pandas", "leidenalg", "celltypist", 
            "infercnvpy", "scipy", 'gseapy', 'anndata', 'scikit-learn', 'statsmodels', 'gseapy']

for pkg in packages:
    print(f"{pkg}: {md.version(pkg)}")

import sys
print(sys.version)

In [ ]:
sc.settings.set_figure_params(dpi_save=300)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def set_plot_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "axes.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "black",
        "grid.color": "#cccccc",
        "grid.linestyle": "--",
        "grid.linewidth": 0.6,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

set_plot_style()

In [ ]:
path = 'file_location'
file_name = ''
adata_gse161529 = sc.read(f"{path}/{file_name}")

In [ ]:
sc.pl.umap(adata_gse161529, color = 'Cell Type')

In [ ]:
sc.pl.umap(adata_gse161529, color = 'Cell Type')
sc.pl.umap(adata_gse161529, color = 'PLAUR', layer = 'scvi_normalized', vmax = 3)
sc.pl.umap(adata_gse161529, color = 'PLAU', layer = 'scvi_normalized', vmax = 3)

In [ ]:
import matplotlib.pyplot as plt
#plt_order = ['B Cell', 'Dendritic', 'Endothelial', 'M-Epithelial', 'Fibroblast', 'Macrophage', 'Myoepithelial', 'N-Epithelial', 'Plasma Cell', 'T Cell']

#Change adata object, keys, and groupby as needed

sc.pl.violin(
    adata_gse161529,
    keys=['PLAUR'], 
    groupby='Cell Type', 
    use_raw=False,
    show=False,
    inner="box",
    jitter=0.4,
    layer='scvi_normalized'
)

# Modify figure
fig = plt.gcf()
fig.set_size_inches(5, 4)
plt.xticks(rotation=90)
plt.ylim(0) 

#plt.xlabel('', labelpad=15)
#plt.ylabel('')  # Change as needed
#plt.title('')

# Save
#plt.savefig("Figures3A,C, 4A-C, 4E-G, Fig5A-C, 5E-F, 6D, 6F.pdf", dpi=300, bbox_inches='tight')

In [ ]:
adata_epith = adata_primary[adata_primary.obs['Cell Type'] == 'Epithelial Cell']
adata_epith = adata_epith[adata_epith.obs['majority_voting'] == 'Epithelial cells']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

gene = "PLAUR"
sample_col = "Sample"
subtype_col = "Subtype"
layer = "scvi_normalized"

subtype_order = ["ER+", "HER2+", "TNBC"]

comparisons = [("HER2+", "TNBC"), ("ER+", "TNBC")]

X = adata_epith[:, gene].layers[layer]

if sp.issparse(X):
    expr = X.toarray().ravel()
else:
    expr = np.asarray(X).ravel()

cell_df = pd.DataFrame({
    sample_col: adata_epith.obs[sample_col].astype(str).values,
    subtype_col: adata_epith.obs[subtype_col].astype(str).values,
    gene: expr
})

sample_df = (
    cell_df
    .groupby(sample_col)
    .agg(
        Subtype=(subtype_col, lambda x: x.mode().iloc[0]),
        mean_expr=(gene, "mean"),
        n_cells=(gene, "size")
    )
    .reset_index()
)

# keep only desired subtypes
sample_df = sample_df[sample_df["Subtype"].isin(subtype_order)].copy()

pairwise_results = []

for g1, g2 in comparisons:
    x = sample_df.loc[sample_df["Subtype"] == g1, "mean_expr"].dropna().values
    y = sample_df.loc[sample_df["Subtype"] == g2, "mean_expr"].dropna().values

    stat, pval = mannwhitneyu(x, y, alternative="less")

    pairwise_results.append({
        "group1": g1,
        "group2": g2,
        "n1": len(x),
        "n2": len(y),
        "mean1": np.mean(x),
        "mean2": np.mean(y),
        "median1": np.median(x),
        "median2": np.median(y),
        "mannwhitney_U": stat,
        "pval": pval
    })

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df["pval_adj_BH"] = multipletests(pairwise_df["pval"], method="fdr_bh")[1]

print(pairwise_df)

plt.figure(figsize=(3, 4.2))

subtype_color_map = {
    "ER+": "#1f77b4",
    "HER2+": "#279e68",
    "TNBC": "#d62728"
}

ax = sns.boxplot(
    data=sample_df,
    x="Subtype",
    y="mean_expr",
    order=subtype_order,
    showfliers=False,
    color="white",
    linewidth=1,
    linecolor="black"
)

sns.stripplot(
    data=sample_df,
    x="Subtype",
    y="mean_expr",
    order=subtype_order,
    hue="Subtype",
    hue_order=subtype_order,
    palette=subtype_color_map,
    size=6,
    jitter=0.15,
    linewidth=1,
    edgecolor="black",
    legend=False,
    ax=ax
)

def format_p(p):
    if p < 0.001:
        return f"p = {p:.1e}"
    return f"p = {p:.3f}"

def add_pval_bar(ax, x1, x2, y, text, h, fontsize=8):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color="black", linewidth=1)
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom", fontsize=fontsize)

# x positions follow subtype_order automatically
xpos = {s: i for i, s in enumerate(subtype_order)}

ymax = sample_df["mean_expr"].max()
yrange = ymax - sample_df["mean_expr"].min()

for i, row in pairwise_df.iterrows():
    y = ymax + (0.06 + 0.10 * i) * yrange
    add_pval_bar(
        ax,
        xpos[row["group1"]],
        xpos[row["group2"]],
        y,
        format_p(row["pval_adj_BH"]),
        h=0.04 * yrange
    )

# give room for annotations
ax.set_ylim(top=ymax + (0.06 + 0.10 * len(pairwise_df) + 0.06) * yrange)
ax.tick_params(axis="both", labelsize=10)
plt.xlabel("")
plt.ylabel("Mean scVI-normalized\n" + rf"$\mathit{{{gene}}}$ expression")
plt.tight_layout()

# plt.savefig(f"Figure4D_{gene}_per_patient_boxplot.pdf", bbox_inches="tight", dpi=300)

plt.show()

In [ ]:
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import itertools
import pandas as pd
import numpy as np

groups = [
    sample_df.loc[sample_df["Subtype"] == subtype, "mean_expr"].dropna().values
    for subtype in subtype_order
]

# only keep groups with at least 2 samples
valid_for_test = [
    subtype for subtype, values in zip(subtype_order, groups)
    if len(values) >= 2
]

groups_valid = [
    sample_df.loc[sample_df["Subtype"] == subtype, "mean_expr"].dropna().values
    for subtype in valid_for_test
]

kruskal_stat, kruskal_p = kruskal(*groups_valid)

print("Kruskal-Wallis test")
print("Groups:", valid_for_test)
print("H =", kruskal_stat)
print("p =", kruskal_p)

In [ ]:
comparisons = [("ER+", "TNBC"), ("HER2+", "TNBC")]

pairwise_results = []

for g1, g2 in comparisons:

    x = sample_df.loc[sample_df["Subtype"] == g1, "mean_expr"].dropna().values
    y = sample_df.loc[sample_df["Subtype"] == g2, "mean_expr"].dropna().values

    stat, pval = mannwhitneyu(x, y, alternative="two-sided")

    pairwise_results.append({
        "group1": g1,
        "group2": g2,
        "n1": len(x),
        "n2": len(y),
        "mean1": np.mean(x),
        "mean2": np.mean(y),
        "median1": np.median(x),
        "median2": np.median(y),
        "mannwhitney_U": stat,
        "pval": pval
    })

pairwise_df = pd.DataFrame(pairwise_results)

pairwise_df["pval_adj_BH"] = multipletests(pairwise_df["pval"], method="fdr_bh")[1]

pairwise_df

In [ ]:
import pandas as pd
import numpy as np

gene = "PLAUR"
layer = "scvi_normalized"

expr = adata_gse161529[:, gene].layers[layer]
expr = expr.A.ravel() if hasattr(expr, "A") else np.asarray(expr).ravel()

df = adata_gse161529.obs[["Sample", "Subtype", "Cell Type"]].copy()
df["Expression"] = expr

df["Subtype"] = df["Subtype"].astype(str).str.strip()
subtype_map = (
    df.groupby("Sample")["Subtype"]
      .agg(lambda x: x.value_counts().index[0])
      .reset_index()
)

df["Group"] = np.where(df["Cell Type"] == "Epithelial Cell", "Epithelial", "Rest")

agg_mean = (
    df.groupby(["Sample", "Group"])["Expression"]
      .mean()
      .reset_index(name="Mean_Expression")
)

agg_counts = (
    df.groupby(["Sample", "Group"])
      .size()
      .reset_index(name="Cell_Count")
)

agg = pd.merge(agg_mean, agg_counts, on=["Sample", "Group"], how="left")

pivot_mean = (
    agg.pivot(index="Sample", columns="Group", values=["Mean_Expression", "Cell_Count"])
    .fillna(0)
    .reset_index()
)

pivot_mean.columns = ['_'.join(col).strip('_') for col in pivot_mean.columns.values]

pivot_mean = pivot_mean.merge(subtype_map, on="Sample", how="left")

pivot_mean["Epi_vs_Rest_Ratio"] = (
    pivot_mean["Mean_Expression_Epithelial"] / pivot_mean["Mean_Expression_Rest"].replace(0, np.nan)
)

pivot_mean = pivot_mean.sort_values("Epi_vs_Rest_Ratio", ascending=False).reset_index(drop=True)

In [ ]:
from scipy.stats import mannwhitneyu

x_tnbc = pivot_mean.loc[pivot_mean["Subtype"] == "TNBC", "Epi_vs_Rest_Ratio"]
x_er   = pivot_mean.loc[pivot_mean["Subtype"] == "ER+", "Epi_vs_Rest_Ratio"]

stat_tnbc_er, pval_tnbc_er = mannwhitneyu(x_tnbc, x_er, alternative="greater")

x_her2 = pivot_mean.loc[pivot_mean["Subtype"] == "HER2+", "Epi_vs_Rest_Ratio"]
stat_tnbc_her2, pval_tnbc_her2 = mannwhitneyu(x_tnbc, x_her2, alternative="greater")

print(f"TNBC vs ER+: U={stat_tnbc_er:.3f}, p={pval_tnbc_er:.4f}")
print(f"TNBC vs HER2+: U={stat_tnbc_her2:.3f}, p={pval_tnbc_her2:.4f}")

In [ ]:
from statsmodels.stats.multitest import multipletests

pvals = [pval_tnbc_er, pval_tnbc_her2]
pvals_adj = multipletests(pvals, method="fdr_bh")[1]

print(f"TNBC vs ER+:   U={stat_tnbc_er:.3f}, p={pval_tnbc_er:.4f}, p_adj={pvals_adj[0]:.4f}")
print(f"TNBC vs HER2+: U={stat_tnbc_her2:.3f}, p={pval_tnbc_her2:.4f}, p_adj={pvals_adj[1]:.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

order = ['ER+', 'HER2+', 'TNBC']

# --- pull color palette from adata_primary ---
subtypes = list(adata_gse161529.obs['Subtype'].cat.categories)
colors   = list(adata_gse161529.uns['Subtype_colors'])
palette  = dict(zip(subtypes, colors))

# --- calculate stats ---
x_tnbc = pivot_mean.loc[pivot_mean["Subtype"] == "TNBC", "Epi_vs_Rest_Ratio"].dropna()
x_er   = pivot_mean.loc[pivot_mean["Subtype"] == "ER+", "Epi_vs_Rest_Ratio"].dropna()
x_her2 = pivot_mean.loc[pivot_mean["Subtype"] == "HER2+", "Epi_vs_Rest_Ratio"].dropna()

stat_tnbc_er, pval_tnbc_er = mannwhitneyu(x_tnbc, x_er, alternative="greater")
stat_tnbc_her2, pval_tnbc_her2 = mannwhitneyu(x_tnbc, x_her2, alternative="greater")

# BH correction across the two comparisons
padj_tnbc_er, padj_tnbc_her2 = multipletests(
    [pval_tnbc_er, pval_tnbc_her2], method="fdr_bh"
)[1]

print(f"TNBC vs ER+:   U={stat_tnbc_er:.1f}, p={pval_tnbc_er:.4g}, p_adj={padj_tnbc_er:.4g}")
print(f"TNBC vs HER2+: U={stat_tnbc_her2:.1f}, p={pval_tnbc_her2:.4g}, p_adj={padj_tnbc_her2:.4g}")

# --- helper function for log-scale p-value bars ---
def add_pval_bar_log(ax, x1, x2, y, text, h_factor=1.08, text_factor=1.03, fontsize=8):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y * h_factor, y * h_factor, y],
        color="black",
        linewidth=1
    )
    ax.text(
        (x1 + x2) / 2,
        y * h_factor * text_factor,
        text,
        ha="center",
        va="bottom",
        fontsize=fontsize
    )

# --- swarm + boxplot ---
fig, ax = plt.subplots(figsize=(3, 3.5))

sns.boxplot(
    data=pivot_mean,
    x="Subtype",
    y="Epi_vs_Rest_Ratio",
    order=order,
    palette=palette,
    showcaps=False,
    boxprops={"facecolor": "none", "zorder": 2},
    whiskerprops={"linewidth": 1.2},
    showfliers=False,
    ax=ax
)

sns.swarmplot(
    data=pivot_mean,
    x="Subtype",
    y="Epi_vs_Rest_Ratio",
    order=order,
    palette=palette,
    size=6,
    alpha=1,
    zorder=3,
    edgecolor="black",
    linewidth=0.6,
    ax=ax
)

# --- formatting before p-value bars ---
ax.set_yscale("log")
ax.axhline(1, color="black", linestyle="--", lw=1)

# --- p-value bar x positions ---
xpos = {
    "ER+": 0,
    "HER2+": 1,
    "TNBC": 2
}

# --- y positions for log scale ---
ymax = pivot_mean["Epi_vs_Rest_Ratio"].max()

y1 = ymax * 1.25
y2 = ymax * 1.95

# --- add p-value bars ---
add_pval_bar_log(
    ax,
    xpos["ER+"],
    xpos["TNBC"],
    y1,
    f"p = {padj_tnbc_er:.4g}"
)

add_pval_bar_log(
    ax,
    xpos["HER2+"],
    xpos["TNBC"],
    y2,
    f"p = {padj_tnbc_her2:.4g}"
)

# give room for annotations
ax.set_ylim(top=y2 * 1.35)

# --- final formatting ---
sns.despine()
ax.set_ylabel(
    r"Mean $\mathit{PLAUR}$ expression"
    "\nM-Epithelial : Stromal"
    "\nlog-scale",
    fontsize=12
)
ax.set_xlabel("")
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.tick_params(axis="both", labelsize=10)
plt.tight_layout()

#plt.savefig("Figure4E_per_patient_PLAUR_epithelial_stromal_ratio.pdf", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.stats import kruskal
import scikit_posthocs as spost

genes_to_plot = ["PLAUR"]
palette = ["#1f77b4", "#279e68", "#d62728"]
order = ["ER+", "HER2+", "TNBC"]

for gene in genes_to_plot:
    expr = (
        adata_gse176_epith[:, gene].layers["scvi_normalized"]
    )

    if sp.issparse(expr):
        expr = expr.toarray().ravel()
    else:
        expr = np.asarray(expr).ravel()

    df = pd.DataFrame({
        "Expression": expr,
        "Subtype": adata_gse176_epith.obs["BC Subtype"].astype(str).str.strip()
    })
    df = df[df["Subtype"].isin(order)].dropna(subset=["Expression", "Subtype"])

    groups = [df.loc[df["Subtype"] == s, "Expression"].values for s in order]
    groups_nonempty = [g for g in groups if len(g) > 0]

    if len(groups_nonempty) >= 2:
        H, p_kw = kruskal(*groups_nonempty)
        kw_text = f"KW: H={H:.2f}, p={p_kw:.2e}"
    else:
        H, p_kw = np.nan, np.nan
        kw_text = "KW: insufficient groups"

    # --- Dunn's posthoc (only if KW is possible) ---
    dunn_text = ""
    dunn_p = None
    if np.isfinite(p_kw) and (df["Subtype"].nunique() >= 2):
        # p-values matrix (rows/cols = groups)
        dunn_p = spost.posthoc_dunn(
            df,
            val_col="Expression",
            group_col="Subtype",
            p_adjust="fdr_bh"   # or "bonferroni", "holm"
        ).reindex(index=order, columns=order)

        # pull the 3 pairwise comparisons you care about
        p_er_her2 = dunn_p.loc["ER+", "HER2+"]
        p_er_tnbc = dunn_p.loc["ER+", "TNBC"]
        p_her2_tnbc = dunn_p.loc["HER2+", "TNBC"]

        dunn_text = (
            f"Dunn (FDR):\n"
            f"ER+ vs HER2+: {p_er_her2:.2e}\n"
            f"ER+ vs TNBC: {p_er_tnbc:.2e}\n"
            f"HER2+ vs TNBC: {p_her2_tnbc:.2e}"
        )

    # --- Plot ---
    plt.figure(figsize=(3, 3.8))
    ax = sns.boxplot(
        data=df,
        x="Subtype", y="Expression",
        order=order,
        showfliers=False,
        whis=(5, 95),
        linewidth=1.2,
        width=0.6,
        palette=palette
    )

    plt.title(f"{gene} (M-Epithelial)", fontsize=14)
    plt.ylabel("scVI-normalized PLAU\n mRNA Expression", fontsize=12)
    plt.xlabel("")
    plt.xticks(rotation=0, fontsize=12)


    plt.tight_layout()
    #plt.savefig("Fig5D.tif", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"{gene}: KW H={H:.3f}, p={p_kw:.3e}")
    if dunn_p is not None:
        print(dunn_p)

In [ ]:
import pandas as pd
import numpy as np

gene = "PLAU"
layer = "scvi_normalized"

expr = adata_gse161529[:, gene].layers[layer]
expr = expr.A.ravel() if hasattr(expr, "A") else np.asarray(expr).ravel()

df = adata_gse161529.obs[["Sample", "Subtype", "Cell Type"]].copy()
df["Expression"] = expr

df["Subtype"] = df["Subtype"].astype(str).str.strip()
subtype_map = (
    df.groupby("Sample")["Subtype"]
      .agg(lambda x: x.value_counts().index[0])
      .reset_index()
)

df["Group"] = np.where(df["Cell Type"] == "Epithelial Cell", "Epithelial", "Rest")

agg_mean = (
    df.groupby(["Sample", "Group"])["Expression"]
      .mean()
      .reset_index(name="Mean_Expression")
)

agg_counts = (
    df.groupby(["Sample", "Group"])
      .size()
      .reset_index(name="Cell_Count")
)

agg = pd.merge(agg_mean, agg_counts, on=["Sample", "Group"], how="left")

pivot_mean = (
    agg.pivot(index="Sample", columns="Group", values=["Mean_Expression", "Cell_Count"])
    .fillna(0)
    .reset_index()
)

pivot_mean.columns = ['_'.join(col).strip('_') for col in pivot_mean.columns.values]

pivot_mean = pivot_mean.merge(subtype_map, on="Sample", how="left")

pivot_mean["Epi_vs_Rest_Ratio"] = (
    pivot_mean["Mean_Expression_Epithelial"] / pivot_mean["Mean_Expression_Rest"].replace(0, np.nan)
)

pivot_mean = pivot_mean.sort_values("Epi_vs_Rest_Ratio", ascending=False).reset_index(drop=True)

In [ ]:
from scipy.stats import mannwhitneyu

x_tnbc = pivot_mean.loc[pivot_mean["Subtype"] == "TNBC", "Epi_vs_Rest_Ratio"]
x_er   = pivot_mean.loc[pivot_mean["Subtype"] == "ER+", "Epi_vs_Rest_Ratio"]

stat_tnbc_er, pval_tnbc_er = mannwhitneyu(x_tnbc, x_er, alternative="greater")

x_her2 = pivot_mean.loc[pivot_mean["Subtype"] == "HER2+", "Epi_vs_Rest_Ratio"]
stat_tnbc_her2, pval_tnbc_her2 = mannwhitneyu(x_tnbc, x_her2, alternative="greater")

print(f"TNBC vs ER+: U={stat_tnbc_er:.3f}, p={pval_tnbc_er:.4f}")
print(f"TNBC vs HER2+: U={stat_tnbc_her2:.3f}, p={pval_tnbc_her2:.4f}")

In [ ]:
from statsmodels.stats.multitest import multipletests

pvals = [pval_tnbc_er, pval_tnbc_her2]
pvals_adj = multipletests(pvals, method="fdr_bh")[1]

print(f"TNBC vs ER+:   U={stat_tnbc_er:.3f}, p={pval_tnbc_er:.4f}, p_adj={pvals_adj[0]:.4f}")
print(f"TNBC vs HER2+: U={stat_tnbc_her2:.3f}, p={pval_tnbc_her2:.4f}, p_adj={pvals_adj[1]:.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

order = ['ER+', 'HER2+', 'TNBC']

# --- pull color palette from adata_primary ---
subtypes = list(adata_gse161529.obs['Subtype'].cat.categories)
colors   = list(adata_gse161529.uns['Subtype_colors'])
palette  = dict(zip(subtypes, colors))

# --- calculate stats ---
x_tnbc = pivot_mean.loc[pivot_mean["Subtype"] == "TNBC", "Epi_vs_Rest_Ratio"].dropna()
x_er   = pivot_mean.loc[pivot_mean["Subtype"] == "ER+", "Epi_vs_Rest_Ratio"].dropna()
x_her2 = pivot_mean.loc[pivot_mean["Subtype"] == "HER2+", "Epi_vs_Rest_Ratio"].dropna()

stat_tnbc_er, pval_tnbc_er = mannwhitneyu(x_tnbc, x_er, alternative="greater")
stat_tnbc_her2, pval_tnbc_her2 = mannwhitneyu(x_tnbc, x_her2, alternative="greater")

# BH correction across the two comparisons
padj_tnbc_er, padj_tnbc_her2 = multipletests(
    [pval_tnbc_er, pval_tnbc_her2], method="fdr_bh"
)[1]

print(f"TNBC vs ER+:   U={stat_tnbc_er:.1f}, p={pval_tnbc_er:.4g}, p_adj={padj_tnbc_er:.4g}")
print(f"TNBC vs HER2+: U={stat_tnbc_her2:.1f}, p={pval_tnbc_her2:.4g}, p_adj={padj_tnbc_her2:.4g}")

# --- helper function for log-scale p-value bars ---
def add_pval_bar_log(ax, x1, x2, y, text, h_factor=1.08, text_factor=1.03, fontsize=8):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y * h_factor, y * h_factor, y],
        color="black",
        linewidth=1
    )
    ax.text(
        (x1 + x2) / 2,
        y * h_factor * text_factor,
        text,
        ha="center",
        va="bottom",
        fontsize=fontsize
    )

# --- swarm + boxplot ---
fig, ax = plt.subplots(figsize=(3, 3.5))

sns.boxplot(
    data=pivot_mean,
    x="Subtype",
    y="Epi_vs_Rest_Ratio",
    order=order,
    palette=palette,
    showcaps=False,
    boxprops={"facecolor": "none", "zorder": 2},
    whiskerprops={"linewidth": 1.2},
    showfliers=False,
    ax=ax
)

sns.swarmplot(
    data=pivot_mean,
    x="Subtype",
    y="Epi_vs_Rest_Ratio",
    order=order,
    palette=palette,
    size=6,
    alpha=1,
    zorder=3,
    edgecolor="black",
    linewidth=0.6,
    ax=ax
)

# --- formatting before p-value bars ---
ax.set_yscale("log")
ax.axhline(1, color="black", linestyle="--", lw=1)

# --- p-value bar x positions ---
xpos = {
    "ER+": 0,
    "HER2+": 1,
    "TNBC": 2
}

# --- y positions for log scale ---
ymax = pivot_mean["Epi_vs_Rest_Ratio"].max()

y1 = ymax * 1.25
y2 = ymax * 1.95

# --- add p-value bars ---
add_pval_bar_log(
    ax,
    xpos["ER+"],
    xpos["TNBC"],
    y1,
    f"p = {padj_tnbc_er:.4g}"
)

add_pval_bar_log(
    ax,
    xpos["HER2+"],
    xpos["TNBC"],
    y2,
    f"p = {padj_tnbc_her2:.4g}"
)

# give room for annotations
ax.set_ylim(top=y2 * 1.35)

# --- final formatting ---
sns.despine()
ax.set_ylabel(
    r"Mean $\mathit{PLAUR}$ expression"
    "\nM-Epithelial : Stromal"
    "\nlog-scale",
    fontsize=12
)
ax.set_xlabel("")
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.tick_params(axis="both", labelsize=10)
plt.tight_layout()

#plt.savefig("Figure5D_per_patient_PLAUR_epithelial_stromal_ratio.pdf", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.stats import kruskal
import scikit_posthocs as spost

genes_to_plot = ["PLAU"]
palette = ["#1f77b4", "#279e68", "#d62728"]
order = ["ER+", "HER2+", "TNBC"]

for gene in genes_to_plot:
    expr = (
        adata_gse176_epith[:, gene].layers["scvi_normalized"]
    )

    if sp.issparse(expr):
        expr = expr.toarray().ravel()
    else:
        expr = np.asarray(expr).ravel()

    df = pd.DataFrame({
        "Expression": expr,
        "Subtype": adata_gse176_epith.obs["BC Subtype"].astype(str).str.strip()
    })
    df = df[df["Subtype"].isin(order)].dropna(subset=["Expression", "Subtype"])

    groups = [df.loc[df["Subtype"] == s, "Expression"].values for s in order]
    groups_nonempty = [g for g in groups if len(g) > 0]

    if len(groups_nonempty) >= 2:
        H, p_kw = kruskal(*groups_nonempty)
        kw_text = f"KW: H={H:.2f}, p={p_kw:.2e}"
    else:
        H, p_kw = np.nan, np.nan
        kw_text = "KW: insufficient groups"

    # --- Dunn's posthoc (only if KW is possible) ---
    dunn_text = ""
    dunn_p = None
    if np.isfinite(p_kw) and (df["Subtype"].nunique() >= 2):
        # p-values matrix (rows/cols = groups)
        dunn_p = spost.posthoc_dunn(
            df,
            val_col="Expression",
            group_col="Subtype",
            p_adjust="fdr_bh"   # or "bonferroni", "holm"
        ).reindex(index=order, columns=order)

        # pull the 3 pairwise comparisons you care about
        p_er_her2 = dunn_p.loc["ER+", "HER2+"]
        p_er_tnbc = dunn_p.loc["ER+", "TNBC"]
        p_her2_tnbc = dunn_p.loc["HER2+", "TNBC"]

        dunn_text = (
            f"Dunn (FDR):\n"
            f"ER+ vs HER2+: {p_er_her2:.2e}\n"
            f"ER+ vs TNBC: {p_er_tnbc:.2e}\n"
            f"HER2+ vs TNBC: {p_her2_tnbc:.2e}"
        )

    # --- Plot ---
    plt.figure(figsize=(3, 3.8))
    ax = sns.boxplot(
        data=df,
        x="Subtype", y="Expression",
        order=order,
        showfliers=False,
        whis=(5, 95),
        linewidth=1.2,
        width=0.6,
        palette=palette
    )

    plt.title(f"{gene} (M-Epithelial)", fontsize=14)
    plt.ylabel("scVI-normalized PLAU\n mRNA Expression", fontsize=12)
    plt.xlabel("")
    plt.xticks(rotation=0, fontsize=12)


    plt.tight_layout()
    plt.savefig("Fig5H_CORRECTED.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"{gene}: KW H={H:.3f}, p={p_kw:.3e}")
    if dunn_p is not None:
        print(dunn_p)

In [ ]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.stats import spearmanr

# -----------------------------
# settings
# -----------------------------
x_gene = "PLAUR"
y_gene = "PLAU"
layer = "scvi_normalized"

# -----------------------------
# helper to extract expression
# -----------------------------
def get_expr(adata, gene, layer):
    X = adata[:, gene].layers[layer]
    if sp.issparse(X):
        return X.toarray().ravel()
    else:
        return np.asarray(X).ravel()

x_expr = get_expr(adata_epith, x_gene, layer)
y_expr = get_expr(adata_epith, y_gene, layer)

df = pd.DataFrame({
    x_gene: x_expr,
    y_gene: y_expr,
    f"log1p_{x_gene}": np.log1p(x_expr),
    f"log1p_{y_gene}": np.log1p(y_expr),
    "Subtype": adata_epith.obs["Subtype"].astype(str).values,
})

draw_order = ["TNBC", "ER+", "HER2+"]

df["Subtype"] = pd.Categorical(
    df["Subtype"],
    categories=draw_order,
    ordered=True
)

df = df.sort_values("Subtype")

plt.figure(figsize=(4, 4))
ax = plt.gca()

sns.scatterplot(
    data=df,
    x=f"log1p_{x_gene}",
    y=f"log1p_{y_gene}",
    hue="Subtype",
    palette=palette,
    alpha=0.8,
    s=10,
    linewidth=0,
    ax=ax,
    legend = False
)

# Linear trend line
sns.regplot(
    data=df,
    x=f"log1p_{x_gene}",
    y=f"log1p_{y_gene}",
    scatter=False,
    color="black",
    line_kws={"linewidth": 1.5},
    ax=ax
)

# -----------------------------
# correlation/statistics
# -----------------------------
from sklearn.linear_model import LinearRegression

r_spearman, p_spearman = spearmanr(df[x_gene], df[y_gene])

x_log = df[f"log1p_{x_gene}"].values.reshape(-1, 1)
y_log = df[f"log1p_{y_gene}"].values

lin_model = LinearRegression()
lin_model.fit(x_log, y_log)
r2 = lin_model.score(x_log, y_log)

ax.text(
    0.05, 0.999,
    f"Linear $R^2$ = {r2:.2f}\nSpearman ρ = {r_spearman:.2f}\np < 0.0001",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
)

# -----------------------------
# aesthetics
# -----------------------------
ax.set_xlabel(r"log$_1p$ normalized $\mathit{PLAUR}$ mRNA expression", fontsize=12)
ax.set_ylabel(r"log$_1p$ normalized $\mathit{PLAU}$ mRNA expression", fontsize=12)
#ax.set_title(r"Per-cell $\mathit{PLAUR}$–$\mathit{PLAU}$ epithelial expression", fontsize=13)
'''
ax.legend(
    title=None,
    loc="upper right",
    bbox_to_anchor=(0.98, 0.98),
    frameon=True,
    facecolor="white",
    framealpha=0.7,
    edgecolor="none",
    fontsize=10,
    markerscale=2
)
'''
sns.despine()
plt.tight_layout()

#plt.savefig("Figure_5I_PLAUR_PLAU_epithelial_mRNA.tif", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np

genes = ["PLAUR", "PLAU"]
layer = "scvi_normalized"

expr_data = {}
for gene in genes:
    expr = adata_epith[:, gene].layers[layer]
    expr = expr.A.ravel() if hasattr(expr, "A") else np.asarray(expr).ravel()
    expr_data[gene] = expr

df = adata_epith.obs[["Sample", "Subtype"]].copy()
for gene in genes:
    df[gene] = expr_data[gene]

df["Subtype"] = df["Subtype"].astype(str).str.strip()
subtype_map = (
    df.groupby("Sample")["Subtype"]
      .agg(lambda x: x.value_counts().index[0])
      .reset_index()
)

agg_mean = (
    df.groupby("Sample")[genes]
      .mean()
      .reset_index()
)

pivot_epith = agg_mean.merge(subtype_map, on="Sample", how="left")

pivot_epith["PLAU_to_PLAUR_Ratio"] = pivot_epith["PLAU"] / pivot_epith["PLAUR"].replace(0, np.nan)

pivot_epith = pivot_epith.sort_values("Subtype").reset_index(drop=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, linregress

# --- clean dataframe ---
plot_df = pivot_epith[["PLAUR", "PLAU", "Subtype"]].dropna().copy()

# --- compute Spearman correlation ---
rho, pval = spearmanr(plot_df["PLAUR"], plot_df["PLAU"])

# --- compute linear regression R² ---
slope, intercept, r_value, lin_pval, stderr = linregress(
    plot_df["PLAUR"],
    plot_df["PLAU"]
)

r2 = r_value**2

# --- color palette from adata_primary ---
subtypes = list(adata_epith.obs["Subtype"].cat.categories)
colors   = list(adata_epith.uns["Subtype_colors"])
palette  = dict(zip(subtypes, colors))

# --- plot ---
plt.figure(figsize=(4, 4))

sns.scatterplot(
    data=plot_df,
    x="PLAUR",
    y="PLAU",
    hue="Subtype",
    palette=palette,
    s=70,
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9
)

# --- regression line across all samples ---
sns.regplot(
    data=plot_df,
    x="PLAUR",
    y="PLAU",
    scatter=False,
    color="black",
    line_kws=dict(lw=1, alpha=0.6)
)

# --- axes formatting ---
plt.xlabel("Mean Epithelial $\it{PLAUR}$ Expression\nper case", fontsize=11)
plt.ylabel("Mean Epithelial $\it{PLAU}$ Expression\nper case", fontsize=11)
#plt.title("Per-Sample Epithelial Expression: PLAUR vs PLAU", fontsize=12, pad=10)

# --- correlation annotation ---
plt.text(
    0.05, 0.99,
    f"Linear $R^2$ = {r2:.2f}\nSpearman ρ = {rho:.2f}\np = {pval:.3g}",
    transform=plt.gca().transAxes,
    fontsize=10,
    ha="left", va="top",
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
)

sns.despine()
plt.grid(alpha=0.55, linestyle="--")
plt.legend(frameon=False, loc="lower right", fontsize=8)
#plt.savefig("Figure6A_Epithelial_PLAUR_PLAU.pdf", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
##Fig6A-C, E
sc.pl.umap(adata_epith, color = 'Cell Type_cnv')
sc.pl.umap(adata_epith, color = 'Subtype')
sc.pl.umap(adata_epith, color = 'PLAUR', layer = 'scvi_normalized', vmax = 4)
sc.pl.umap(adata_epith, color = 'PLAU', layer = 'scvi_normalized', vmax = 2)

In [ ]:
import scipy

In [ ]:
plaured_normalized_expr = adata_tnbc[:, 'PLAUR'].layers['scvi_normalized']

if scipy.sparse.issparse(plaured_normalized_expr):
    plaured_normalized_expr = plaured_normalized_expr.toarray().flatten()
else:
    plaured_normalized_expr = plaured_normalized_expr.flatten()

plaured_90th_percentile = np.percentile(plaured_normalized_expr, 90)
plaured_10th_percentile = np.percentile(plaured_normalized_expr, 10)

plaured_status = np.where(plaured_normalized_expr >= plaured_90th_percentile, 'PLAUR+',
                          np.where(plaured_normalized_expr <= plaured_10th_percentile, 'PLAUR-', 'Intermediate'))

adata_tnbc.obs['PLAUR Status'] = plaured_status

In [ ]:
custom_colors = {
    "PLAUR+": "#BA1F07",   # deep red
    "PLAUR-": "#026EA8",   # dark blue
    "Intermediate": "#b8b8b8"    # gray
}


import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

# enforce desired order
plaur_order = ["PLAUR-", "Intermediate", "PLAUR+"]

adata_tnbc.obs["PLAUR Status"] = pd.Categorical(
    adata_tnbc.obs["PLAUR Status"],
    categories=plaur_order,
    ordered=True
)

In [ ]:
EMT = [
    'CXCL8', 'SERPINE2', 'SDC4', 'SERPINE1', 'LAMA3',
    'TNFRSF11B', 'LAMC2', 'CXCL1', 'SAT1', 'PTHLH',
    'COMP', 'NT5E', 'LGALS1', 'CALD1', 'SPP1',
    'TIMP3', 'PMEPA1', 'SPOCK1', 'QSOX1', 'PDLIM4',
    'IL32', 'TNFRSF12A', 'TPM4', 'GADD45A', 'MMP1',
    'LUM', 'IGFBP3', 'MMP2', 'ITGA2', 'TPM1',
    'MMP3', 'FN1', 'VEGFC', 'INHBA', 'VEGFA',
    'GREM1', 'SDC1', 'TGFBI', 'ITGA5', 'CD44'
]

HYPOXIA = [
    'ERO1A', 'PPP1R15A', 'CDKN1A', 'BTG1', 'SDC4',
    'SLC2A1', 'STC1', 'ADM', 'IRS2', 'SLC2A3',
    'ENO1', 'NDRG1', 'HK2', 'AKAP12', 'MT2A',
    'LDHA', 'HAS1', 'PIM1', 'PGK1', 'IDS',
    'IER3', 'CA12', 'BNIP3L', 'TPI1', 'ANXA2',
    'DUSP1', 'IGFBP3'
]

CIN_Bakhoum = [
    'PELI2', 'BMP2', 'SHH', 'TNS4', 'RAB3B',
    'ROBO1', 'ARHGAP28', 'CHN2', 'CST1', 'F13A1',
    'CPVL', 'SEMA6D', 'NHSL2', 'GTF2IP7', 'DPYSL3',
    'PCDH7', 'KHDRBS3', 'TRAC', 'TMEM156', 'CST4',
    'CD24', 'FGF5', 'NTN4', 'EZH2', 'JUN',
    'VIM', 'STEAP1', 'SOX4', 'MMP14', 'TIMP1',
    'ZEB1'
]

TAI = [
    'CDKN1A', 'BTG1', 'CSF2', 'IRS2', 'CXCL1',
    'SLC2A3', 'CXCL3', 'SAT1', 'CXCL2', 'ICAM1',
    'OLR1', 'PMEPA1', 'PHLDA2', 'IER3', 'TGIF1',
    'DUSP5', 'SERPINB2', 'DUSP1', 'FOS', 'F3',
    'IL1A', 'IL1B', 'SGK1', 'ATF3', 'CD44',
    'HBEGF', 'PPP1R15A', 'CEBPB', 'SDC4', 'PTGS2',
    'CCL5', 'CCL4', 'CCL2', 'EGR1', 'LAMB3',
    'GADD45A', 'CCL20', 'G0S2', 'INHBA', 'SOD2',
    'PNRC1', 'EIF1', 'VEGFA', 'FOSL1', 'NFKBIA',
    'RCAN1', 'BMP2', 'KLF6', 'ACKR3', 'MXD1',
    'CSF3', 'CXCL8', 'AQP9', 'C5AR1', 'ADM',
    'GNA15', 'CCL7', 'PDPN', 'CLEC5A', 'ITGB8',
    'CD55', 'TNFRSF1B', 'SELENOS', 'ITGA5', 'CHST2'
]
sc.tl.score_genes(adata_tnbc, gene_list= EMT, score_name='EMT', use_raw=False, layer = 'scvi_normalized')
sc.tl.score_genes(adata_tnbc, gene_list= HYPOXIA, score_name='Hypoxia', use_raw=False, layer = 'scvi_normalized')
sc.tl.score_genes(adata_tnbc, gene_list= CIN_Bakhoum, score_name='CIN', use_raw=False, layer = 'scvi_normalized')
sc.tl.score_genes(adata_tnbc, gene_list= TAI, score_name='TAI', use_raw=False, layer = 'scvi_normalized')

In [ ]:
##Replace keys with Signature Score Of Interest

sc.pl.violin(
    adata_tnbc,
    keys=["EMT"],
    groupby="PLAUR Status",
    use_raw=False,
    inner="box",
    palette = custom_colors,
    jitter=0,
    order = ['PLAUR-', 'Intermediate', 'PLAUR+'],
    layer="scvi_normalized",
    show=False
)

# Modify figure
fig = plt.gcf()
fig.set_size_inches(3, 4)
plt.xticks(rotation=90)

plt.xlabel("")
#plt.ylabel('Module Score')
)
#plt.savefig("Fig6G-J.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -----------------------------
# settings
# -----------------------------
adata_use = adata_tnbc
score = "EMT"            # signature score column in .obs
group_col = "PLAUR Status"
groups = ["PLAUR-", "Intermediate", "PLAUR+"]

# -----------------------------
# data
# -----------------------------
df = adata_use.obs[[group_col, score]].copy()
vals = {g: df.loc[df[group_col] == g, score].dropna().values for g in groups}

# -----------------------------
# overall test: Kruskal-Wallis + epsilon-squared
# -----------------------------
H, p_kw = kruskal(*vals.values())
n_total = sum(len(v) for v in vals.values())
eps2 = H / (n_total - 1)

print(f"Signature: {score}")
print(f"Kruskal-Wallis: H = {H:.2f}, p = {p_kw:.3g}, epsilon² = {eps2:.3f}  (n = {n_total} cells)\n")

# -----------------------------
# pairwise Mann-Whitney U + effect sizes
# -----------------------------
rows = []
for g1, g2 in itertools.combinations(groups, 2):
    x, y = vals[g1], vals[g2]
    U, p = mannwhitneyu(x, y, alternative="two-sided")

    rank_biserial = 2 * U / (len(x) * len(y)) - 1     # >0 means g1 tends to be higher

    rows.append({
        "signature": score,
        "group1": g1,
        "group2": g2,
        "n1": len(x),
        "n2": len(y),
        "median1": np.median(x),
        "median2": np.median(y),
        "median_diff": np.median(x) - np.median(y),
        "rank_biserial_r": rank_biserial,
        "U": U,
        "pval": p,
    })

pw_cells = pd.DataFrame(rows)
pw_cells["pval_adj_BH"] = multipletests(pw_cells["pval"], method="fdr_bh")[1]

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print(pw_cells)

In [ ]:
##DEG

In [ ]:
adata_plaur_tnbc = adata_tnbc[adata_tnbc.obs['PLAUR Status'] != 'Intermediate']
adata_plaur_tnbc.obs['PLAUR Status'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata_plaur_tnbc, 'PLAUR Status', use_raw=False, layer = 'scvi_normalized')
sc.pl.rank_genes_groups(adata_plaur_tnbc, n_genes=20, sharey=False)
markers = sc.get.rank_genes_groups_df(adata_plaur_tnbc, None)
markers = markers[(markers.pvals_adj < 0.05) & (markers.logfoldchanges > 0)]
markers

In [ ]:
import pandas as pd
import mygene

# Load and clean
df = markers
df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

# Filter log2FC > 2
filtered = df[df['logfoldchanges'] > 2]

enriched_names_plaur = filtered.loc[
    filtered['group'] == 'PLAUR+', 'names'
].tolist()

enriched_names_plaur_down = filtered.loc[
    filtered['group'] == 'PLAUR-', 'names'
].tolist()

# Query protein-coding genes
mg = mygene.MyGeneInfo()

my_genes = list(set(enriched_names_plaur + enriched_names_plaur_down))

results = mg.querymany(
    my_genes,
    scopes='symbol',
    fields='type_of_gene',
    species='human'
)

protein_coding = {
    res['query'] for res in results
    if res.get('type_of_gene') == 'protein-coding'
}

# Final gene lists
tnbc_plaur_deg = [
    gene for gene in enriched_names_plaur if gene in protein_coding
]

tnbc_plaur_deg_down = [
    gene for gene in enriched_names_plaur_down if gene in protein_coding
]

print(f"PLAUR+ protein-coding genes: {len(tnbc_plaur_deg)}")
print(f"PLAUR- protein-coding genes: {len(tnbc_plaur_deg_down)}")

In [ ]:
###DEGs

In [ ]:
heatmap_genes = ['ANXA1', 'B2M', 'BCAT1', 'CALD1', 'CCL2', 'CCL20', 'CCNA1', 'CD93', 'CDA', 'CDH13', 
                     'CHI3L1', 'CSTB', 'CXCL3', 'CXCL5', 'EPHA2', 'FABP5', 'FCGR2B', 'FGF1', 'FN1', 'GJA5', 
                     'GJB2', 'GREM1', 'HAS1', 'HLA-A', 'HLA-B', 'ICAM1', 'IDO1', 'IGFBP3', 'ITGB8', 'KCNJ15', 
                     'KLK7', 'KRT17', 'KRT6A', 'KRT6B', 'LAMB3', 'LAMC2', 'LCN2', 'LEMD1', 'LMNA', 'LUM', 'LY6D', 
                     'MMP2', 'MMP3', 'MMP7', 'MYL12A', 'NFKBIA', 'NT5E', 'OLR1', 'PDPN', 'PGBD5', 'PI3', 'PLAUR', 
                     'PLAU', 'PLS3', 'PTGS2', 'RAET1L', 'RARRES1', 'RHCG', 'RNASE1', 'S100A10', 'S100A14', 
                     'S100A16', 'S100A2', 'S100A4', 'S100A6', 'SAA1', 'SERPINB5', 'SERPINE1', 'SERPINE2', 
                     'SGK1', 'SOD2', 'SPOCD1', 'SPP1', 'SRGN', 'TACSTD2', 'TEK', 'TGFBI', 'TMEM45A', 
                     'TNFRSF12A', 'TREM1', 'TSHZ3', 'WFDC2', 'YBX3']

In [ ]:
import pandas as pd

df_161529 = pd.read_csv("markers_TNBC_PLAUR_GSE161529.csv")
df_246613 = pd.read_csv("markers_TNBC_PLAUR_GSE246613.csv")
df_176078 = pd.read_csv("markers_TNBC_PLAUR_GSE176078.csv")

In [ ]:
for df in [df_161529, df_246613, df_176078]:
    df.rename(columns={"names": "gene"}, inplace=True)

genes = sorted(heatmap_genes)  # keep your existing order (or sorted(heatmap_genes))

def get_logfc(df, genes):
    s = df.set_index("gene")["logfoldchanges"]
    # pull in YOUR order; will error if any gene is missing
    return s.loc[genes]
fc_161529 = get_logfc(df_161529, heatmap_genes)
fc_246613 = get_logfc(df_246613, heatmap_genes)
fc_176078 = get_logfc(df_176078, heatmap_genes)

In [ ]:
contrast = pd.DataFrame({
    "GSE161529 PLAUR-": -fc_161529,
    "GSE246613 PLAUR-": -fc_246613,
    "GSE176078 PLAUR-": -fc_176078,
    "GSE161529 PLAUR+":  fc_161529,
    "GSE246613 PLAUR+":  fc_246613,
    "GSE176078 PLAUR+":  fc_176078,
})

In [ ]:
vals = contrast.values
genes = contrast.index
conds = contrast.columns

fig, ax = plt.subplots(figsize=(4, 20))

im = ax.imshow(
    vals,
    aspect="auto",
    cmap="Reds",      # or RdBu_r if signed
    vmin=-4, vmax=4
)

ax.set_xlim(-0.5, vals.shape[1] - 0.5)
ax.set_ylim(vals.shape[0] - 0.5, -0.5)

ax.set_xticks(range(vals.shape[1]))
ax.set_xticklabels(conds, rotation=90, ha="center")
ax.set_yticks(range(vals.shape[0]))
ax.set_yticklabels(genes)

ax.set_xticks(np.arange(-0.5, vals.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, vals.shape[0], 1), minor=True)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=.75
)

ax.tick_params(which="minor", bottom=False, left=False)

# colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.2)
cbar.set_label("log2FC")

ax.set_title("Upregulated Genes")
plt.tight_layout()
plt.grid(False)
plt.savefig("Fig7A.tif", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
genes = sorted(plaur_tnbc_x3_down)

In [ ]:
def get_logfc(df, genes):
    s = df.set_index("gene")["logfoldchanges"]
    return s.loc[genes]

fc_161529_down = get_logfc(df_161529, genes)
fc_246613_down = get_logfc(df_246613, genes)
fc_176078_down = get_logfc(df_176078, genes)

In [ ]:
contrast = pd.DataFrame({
    "GSE161529 PLAUR-": fc_161529_down,
    "GSE246613 PLAUR-": fc_246613_down,
    "GSE176078 PLAUR-": fc_176078_down,
    "GSE161529 PLAUR+": -fc_161529_down,
    "GSE246613 PLAUR+": -fc_246613_down,
    "GSE176078 PLAUR+": -fc_176078_down,
})

In [ ]:
contrast = contrast[~contrast.index.str.startswith("MT-")]

In [ ]:
vals = contrast.values
genes = contrast.index
conds = contrast.columns

fig, ax = plt.subplots(figsize=(4, 11))

im = ax.imshow(
    vals,
    aspect="auto",
    cmap="Blues",      # or RdBu_r if signed
    vmin=-5, vmax=5
)

ax.set_xlim(-0.5, vals.shape[1] - 0.5)
ax.set_ylim(vals.shape[0] - 0.5, -0.5)

ax.set_xticks(range(vals.shape[1]))
ax.set_xticklabels(conds, rotation=90, ha="center")
ax.set_yticks(range(vals.shape[0]))
ax.set_yticklabels(genes)

ax.set_xticks(np.arange(-0.5, vals.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, vals.shape[0], 1), minor=True)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=1
)

ax.tick_params(which="minor", bottom=False, left=False)

# colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.4)
cbar.set_label("log2FC (PLAUR+ vs PLAUR−)")

ax.set_title("Upregulated Genes")
plt.tight_layout()
plt.grid(False)
#plt.savefig("Fig7B.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
###GSEA

In [ ]:
import pandas as pd
import gseapy as gp

def run_signed_plaur_gsea(csv_path, subtype_name):
    # 1) Read CSV
    df = pd.read_csv(csv_path)

    # 2) Sanity check
    assert {"PLAUR+", "PLAUR-"}.issubset(set(df["group"].unique())), \
        f"{csv_path} does not contain both PLAUR+ and PLAUR-"

    # 3) Create signed logFC
    #    PLAUR+  -> positive
    #    PLAUR-  -> negative
    df["signed_logFC"] = df["logfoldchanges"]
    df.loc[df["group"] == "PLAUR-", "signed_logFC"] *= -1

    # 4) Build ranked list
    rnk = (
        df[["names", "signed_logFC"]]
        .rename(columns={"names": "gene", "signed_logFC": "rank"})
        .sort_values("rank", ascending=False)
    )

    # 5) Run preranked GSEA
    pre = gp.prerank(
        rnk=rnk,
        gene_sets="MSigDB_Hallmark_2020",
        min_size=10,
        max_size=500,
        permutation_num=5000,
        seed=7,
        outdir=None,
    )

    # 6) Collect results
    res = pre.res2d.copy()
    res["Subtype"] = subtype_name

    return res, pre

In [ ]:
res_tnbc, pre_tnbc = run_signed_plaur_gsea(
    "markers_TNBC_PLAUR_GSE161529.csv", "GSE161529"
)

res_her2, pre_her2 = run_signed_plaur_gsea(
    "markers_TNBC_PLAUR_GSE246613.csv", "GSE246613"
)

res_er, pre_er = run_signed_plaur_gsea(
    "markers_TNBC_PLAUR_GSE176078.csv", "GSE176078"
)

all_res = pd.concat([res_tnbc, res_her2, res_er], ignore_index=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- make sure all_res exists ---
try:
    all_res
except NameError:
    all_res = pd.concat([res_tnbc, res_her2, res_er], ignore_index=True)

df = all_res.copy()

df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")
df = df.dropna(subset=["NES", "FDR q-val", "Term", "Subtype"])

# --- pivot ALL terms ---
nes = df.pivot_table(index="Term", columns="Subtype", values="NES", aggfunc="mean")
fdr = df.pivot_table(index="Term", columns="Subtype", values="FDR q-val", aggfunc="mean")

# ensure float
nes = nes.apply(pd.to_numeric, errors="coerce").astype(float)
fdr = fdr.apply(pd.to_numeric, errors="coerce").astype(float)

CMAP = "inferno"
STAR_FDR = 0.25
FIG_W = 4
FIG_H = None

REQUIRE_COMPLETE = True
VMIN = -2
VMAX = 2
SYMMETRIC_AROUND_ZERO = False


if REQUIRE_COMPLETE:
    nes = nes.dropna(axis=0, how="any")
    fdr = fdr.loc[nes.index]

# optional ordering
nes = nes.sort_index()
fdr = fdr.loc[nes.index]


desired_order = ["GSE161529", "GSE246613", "GSE176078"]

# (optional) debug: see what your actual column labels are
print("Actual nes columns:", list(nes.columns))

# keep only columns that exist (prevents KeyError)
desired_order = [c for c in desired_order if c in nes.columns]

nes = nes.loc[:, desired_order]
fdr = fdr.loc[:, desired_order]

print("Plotted column order:", list(nes.columns))

# ----------------------------
# COLOR LIMITS
# ----------------------------
vals = nes.values.astype(float)

if SYMMETRIC_AROUND_ZERO:
    m = np.nanmax(np.abs(vals))
    vmin, vmax = (-m, m)
else:
    vmin = VMIN if VMIN is not None else np.nanmin(vals)
    vmax = VMAX if VMAX is not None else np.nanmax(vals)

if FIG_H is None:
    FIG_H = max(7, 0.26 * len(nes))

# --- plot ---
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
im = ax.imshow(vals, aspect="auto", cmap=CMAP, vmin=vmin, vmax=vmax)

ax.set_xlim(-0.5, nes.shape[1] - 0.5)
ax.set_ylim(nes.shape[0] - 0.5, -0.5)

ax.set_xticks(range(nes.shape[1]))
ax.set_xticklabels(nes.columns, rotation=90, ha="center")
ax.set_yticks(range(nes.shape[0]))
ax.set_yticklabels(nes.index)

ax.set_xticks(np.arange(-0.5, nes.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, nes.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("NES")

for i in range(nes.shape[0]):
    for j in range(nes.shape[1]):
        q = fdr.iloc[i, j]
        val = nes.iloc[i, j]
        if np.isfinite(q) and q < STAR_FDR:
            is_neg = val < 0
            ax.scatter(
                j, i,
                marker="o",
                s=20,
                c="white" if is_neg else "black",
                edgecolors="black" if is_neg else "none",
                linewidths=0.5 if is_neg else 0,
                zorder=10
            )
ax.grid(False)

plt.tight_layout()
plt.savefig("Fig7C.pdf", dpi=300, bbox_inches="tight")
plt.show()